# 03 · Routing with LangChain

Routing sends each input to a **different** chain based on its content. Three approaches, from simplest to smartest:

1. **Custom function router** — plain Python `if/else` returning a chain.
2. **LLM classifier + `RunnableBranch`** — let a model pick the category.
3. **Semantic routing** — route by embedding similarity, no extra LLM call.

---

## Setup

In [2]:
%pip install -qU langchain langchain-openai python-dotenv numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from langchain.chat_models import init_chat_model
from langchain.embeddings import init_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from dotenv import load_dotenv

load_dotenv("../.env")

# Model comes from .env in "provider:model" form (e.g. openai:gpt-4o-mini),
# so switching models is a config change, not a code change.
llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0.3)
parser = StrOutputParser()

## The destination chains

A support desk with three specialists. Each is a normal LCEL chain — routing just decides which one runs.

In [4]:
billing_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a billing specialist. Be precise about money, refunds, and invoices."),
        ("human", "{query}"),
    ]) | llm | parser
)

tech_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a technical support engineer. Give clear, step-by-step fixes."),
        ("human", "{query}"),
    ]) | llm | parser
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a friendly support generalist. Help or politely redirect."),
        ("human", "{query}"),
    ]) | llm | parser
)

---
## 1. Custom function router

The simplest router: a function that inspects the input and returns the chain to run. Wrap it so the chosen chain actually executes.

In [5]:
def keyword_router(inputs: dict):
    q = inputs["query"].lower()
    if any(w in q for w in ("refund", "invoice", "charge", "payment", "billing")):
        return billing_chain
    if any(w in q for w in ("error", "crash", "bug", "install", "login")):
        return tech_chain
    return general_chain

# RunnableLambda returns a chain; LangChain then invokes it with the same input.
rule_router = RunnableLambda(keyword_router)

print(rule_router.invoke({"query": "I was charged twice, I need a refund"})[:200])

I apologize for the inconvenience. Please provide your invoice number or the date of the charge so I can locate the duplicate transaction and process your refund promptly.


> Fast and free, but brittle: "my card got hit twice" has no keyword. That's where the LLM classifier helps.

---
## 2. LLM classifier + RunnableBranch

Step 1: an LLM labels the query. Step 2: `RunnableBranch` (an if/elif/else over runnables) picks the chain from that label.

In [6]:
classifier = (
    ChatPromptTemplate.from_template(
        "Classify the support query into exactly one word: "
        "'billing', 'technical', or 'general'.\n\nQuery: {query}\n\nCategory:"
    ) | llm | parser
)

# Attach the classification, keeping the original query around.
from langchain_core.runnables import RunnablePassthrough

branch = RunnableBranch(
    (lambda x: "billing" in x["category"].lower(), billing_chain),
    (lambda x: "technical" in x["category"].lower(), tech_chain),
    general_chain,  # default
)

llm_router = RunnablePassthrough.assign(category=classifier) | branch

In [7]:
for q in [
    "My card got hit twice this month",
    "The app crashes every time I open settings",
    "What are your support hours?",
]:
    print(f"Q: {q}")
    print(f"A: {llm_router.invoke({'query': q})[:160]}\n")

Q: My card got hit twice this month
A: I'm sorry to hear about the double charge. Please provide your account or invoice number so I can locate the transactions and process a refund if necessary.

Q: The app crashes every time I open settings
A: I'm sorry to hear you're experiencing this issue. Let's try some troubleshooting steps to resolve the app crashing when you open the settings:

### Step 1: Rest

Q: What are your support hours?
A: Hello! I'm here to help you anytime you need. If you're referring to specific support hours for a service or company, please let me know which one, and I'll do 



---
## 3. Semantic routing (embeddings)

Skip the classifier LLM call entirely. Embed a few example utterances per route once, then route a query to the route whose examples it's most similar to. Cheaper and faster than an LLM classifier.

In [8]:
import numpy as np

embeddings = init_embeddings(
    os.environ["EMBEDDING_MODEL"],
    provider=os.environ["MODEL_PROVIDER"],
)

route_examples = {
    "billing": [
        "I want a refund", "my invoice is wrong", "why was I charged twice",
        "cancel my subscription and refund me",
    ],
    "technical": [
        "the app keeps crashing", "I get an error on login",
        "installation fails", "the page won't load",
    ],
    "general": [
        "what are your hours", "where are you located",
        "do you have a mobile app", "how do I contact a human",
    ],
}

# Precompute a centroid embedding per route.
route_centroids = {
    name: np.mean(embeddings.embed_documents(examples), axis=0)
    for name, examples in route_examples.items()
}

chains_by_route = {"billing": billing_chain, "technical": tech_chain, "general": general_chain}

In [9]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def semantic_router(inputs: dict):
    q_vec = embeddings.embed_query(inputs["query"])
    best = max(route_centroids, key=lambda r: cosine(q_vec, route_centroids[r]))
    print(f"  ↳ routed to: {best}")
    return chains_by_route[best]

semantic_route = RunnableLambda(semantic_router)

for q in [
    "can you reverse the duplicate payment",
    "app throws a 500 when I sign in",
    "is there a dark mode",
]:
    print(f"Q: {q}")
    print(f"A: {semantic_route.invoke({'query': q})[:140]}\n")

Q: can you reverse the duplicate payment
  ↳ routed to: billing
A: Yes, I can process a refund for the duplicate payment. Please provide the invoice number or payment details so I can locate the transaction 

Q: app throws a 500 when I sign in
  ↳ routed to: technical
A: A 500 Internal Server Error typically indicates a problem on the server side. Here are step-by-step troubleshooting steps to identify and re

Q: is there a dark mode
  ↳ routed to: general
A: Hello! If you're asking about enabling dark mode, it depends on the specific app or device you're using. Could you please tell me which plat



---
## What to remember

| Approach | How it decides | Cost | Best when |
|---|---|---|---|
| Function router | your Python logic | free | clear keywords / structured input |
| LLM + `RunnableBranch` | a model classifies | 1 extra LLM call | fuzzy language, few routes |
| Semantic routing | embedding similarity | 1 cheap embed call | many routes, low latency, no keywords |

| Tool | Role |
|---|---|
| `RunnableLambda(fn)` | wrap a function that returns the chain to run |
| `RunnableBranch((cond, chain), ..., default)` | if/elif/else over runnables |
| `RunnablePassthrough.assign(category=...)` | add the classification, keep the query |

**Next:** Article 04 — **Orchestrator–worker**: a planner LLM splits work into subtasks and delegates to specialists.